第 1 步：先做一个最小 PyTorch 模型

In [ ]:
import torch
import torch.nn as nn

class MiniModel(nn.Module):
    def forward(self, x):
        y = torch.relu(x)
        z = y + 1.0
        return z

model = MiniModel().eval()
x = torch.tensor([[-1.0, 0.5, 2.0]], dtype=torch.float32)

torch.onnx.export(
    model,
    x,
    "work/mini.onnx",
    input_names=["input"],
    output_names=["output"],
    opset_version=13
)

print("exported: work/mini.onnx")

把 ONNX 里的 Add 改成 custom op

In [2]:
import onnx
from onnx import helper

model = onnx.load("work/mini.onnx")
graph = model.graph

new_nodes = []
for node in graph.node:
    if node.op_type == "Add":
        custom_node = helper.make_node(
            "MyScale",
            inputs=[node.input[0]],
            outputs=list(node.output),
            domain="my.custom",
            alpha=2.0
        )
        new_nodes.append(custom_node)
    else:
        new_nodes.append(node)

del graph.node[:]
graph.node.extend(new_nodes)

# 清理未被引用的 Constant 节点
used_inputs = set()
for node in graph.node:
    used_inputs.update(node.input)

filtered_nodes = []
for node in graph.node:
    if node.op_type == "Constant" and all(o not in used_inputs for o in node.output):
        continue
    filtered_nodes.append(node)

del graph.node[:]
graph.node.extend(filtered_nodes)

model.opset_import.extend([helper.make_opsetid("my.custom", 1)])
onnx.save(model, "work/mini_custom.onnx")
print("saved: work/mini_custom.onnx")

saved: work/mini_custom.onnx


我们规定：

MyScale(x, alpha) = x * alpha

所以整个模型变成：

output = MyScale(Relu(x), alpha=2.0)

如果输入：

[-1.0, 0.5, 2.0]

那么：

Relu -> [0.0, 0.5, 2.0]
MyScale -> [0.0, 1.0, 4.0]

开始op-package写 OpDef XML
写 OpDef XML

你贴的文档已经说明了，OpDef 里至少要有：

Name
Input
Output
Parameter（可选）
SupportedBackend
UseDefaultTranslation

并且 custom op 至少要有一个输入和一个输出

这里的关键点：

Reference Source="ONNX"：给 converter 做 source-side 对应
UseDefaultTranslation=false：表示这是 generic custom op，不是覆盖 QNN 原生 op
SupportedBackend=CPU：先走最小闭环，不碰 HTP

In [ ]:
<OpDefCollection PackageName="MyScaleOpPackage">

  <OpDef>
    <Name>MyScale</Name>

    <Description>
      <Content>Multiply input tensor by scalar alpha</Content>
    </Description>

    <Reference Source="ONNX" Url="custom://my.custom/MyScale"></Reference>

    <Input>
      <Name>input</Name>
      <Mandatory>true</Mandatory>
      <Datatype>QNN_DATATYPE_FLOAT_32</Datatype>
      <Shape>
        <Rank>ND</Rank>
        <Layout>UNDEFINED</Layout>
      </Shape>
    </Input>

    <Output>
      <Name>output</Name>
      <Mandatory>true</Mandatory>
      <Datatype>QNN_DATATYPE_FLOAT_32</Datatype>
      <Shape>
        <Rank>ND</Rank>
        <Layout>UNDEFINED</Layout>
      </Shape>
    </Output>

    <Parameter>
      <Name>alpha</Name>
      <Mandatory>true</Mandatory>
      <Datatype>QNN_DATATYPE_FLOAT_32</Datatype>
      <Shape>
        <Rank>SCALAR</Rank>
        <Layout>UNDEFINED</Layout>
      </Shape>
      <Default>1.0</Default>
    </Parameter>

    <UseDefaultTranslation>false</UseDefaultTranslation>
    <SupportedBackend>CPU</SupportedBackend>
  </OpDef>

</OpDefCollection>

生成 custom op package 骨架

QNN/QAIRT 文档里有 qnn-op-package-generator 的说明，用来根据 op package 配置生成包骨架。

In [12]:
import subprocess

subprocess.run([
    "qnn-op-package-generator",
    "-p", "custom_op/custom_Op.xml",
    "--output_path", "work/op_package"
], check=True)

2026-04-05 11:17:21,023 - 278 - INFO - Note: Interface provider function will be named: MyScaleOpPackageInterfaceProvider
2026-04-05 11:17:21,035 - 278 - INFO - Code generation is complete for package: MyScaleOpPackage at /workspace/work/op_package


CompletedProcess(args=['qnn-op-package-generator', '-p', 'custom_op/custom_Op.xml', '--output_path', 'work/op_package'], returncode=0)

修改src下的ops，实现
所以实现一个op 我要关注的就是 权重怎么读进来 输入怎么读取 输出怎么输出 这几个主要的 就够了把
基本理解是对的，这三件是核心主线，但还需要再补几项“容易踩坑但必须做”的点，才算一个可用的 op。
最小完整清单建议是：

参数/权重读取
包括参数名、类型、shape、默认值、是否必填。

输入读取
校验输入个数、dtype、维度、layout、量化信息（如果有）。

输出写回
输出 shape 推导、dtype 一致性、内存可写性、边界检查。

配置校验
在创建或 finalize 阶段把不合法配置尽早拦掉，避免 execute 才崩。

生命周期管理
初始化阶段缓存元数据，执行阶段只做计算；释放阶段把申请的资源清干净。

错误码和健壮性
空指针、长度不匹配、类型不匹配都要返回明确错误码，不要静默失败。

性能与一致性
避免在 execute 里重复做昂贵解析；多线程/重复调用场景行为一致。

In [14]:
import subprocess

subprocess.run([
    "make",
    "-C", "work/op_package/MyScaleOpPackage",
    "cpu_x86"
], check=True)

make: Entering directory '/workspace/work/op_package/MyScaleOpPackage'
make -f makefiles/Makefile.linux-x86_64
make[1]: Entering directory '/workspace/work/op_package/MyScaleOpPackage'
Copying custom op source files from SDK
clang++ -std=c++11 -fno-exceptions -fPIC -pg -I/opt/qairt/2.44.0.260225/include/QNN -I include -I/opt/qairt/2.44.0.260225/include/QNN/CPU -I /opt/qairt/2.44.0.260225/share/QNN/OpPackageGenerator/CustomOp -I src/utils -I src/utils/CPU -march=x86-64 -O3 -Wno-write-strings -fvisibility=hidden -DQNN_API="__attribute__((visibility(\"default\")))" -c src/CpuCustomOpPackage.cpp -o obj/x86_64-linux-clang/CpuCustomOpPackage.o
clang++ -std=c++11 -fno-exceptions -fPIC -pg -I/opt/qairt/2.44.0.260225/include/QNN -I include -I/opt/qairt/2.44.0.260225/include/QNN/CPU -I /opt/qairt/2.44.0.260225/share/QNN/OpPackageGenerator/CustomOp -I src/utils -I src/utils/CPU -march=x86-64 -O3 -Wno-write-strings -fvisibility=hidden -DQNN_API="__attribute__((visibility(\"default\")))"  -shared o

clang: warning: argument unused during compilation: '-pg' [-Wunused-command-line-argument]


CompletedProcess(args=['make', '-C', 'work/op_package/MyScaleOpPackage', 'cpu_x86'], returncode=0)

先尝不添加自定义op 转换qnn

In [15]:
subprocess.run([
    "qnn-onnx-converter",
    "--input_network", "work/mini_custom.onnx",
    "--output_path", "work/mini_custom.cpp"
], check=True)

2026-04-05 11:40:42,259 - 278 - INFO - Input shape info 
2026-04-05 11:40:43,647 - 283 - WARNING - Symbolic shape inference Failed. Exception: Incomplete symbolic shape inference.. Running normal shape inference.
2026-04-05 11:40:44,736 - 283 - WARNING - WARNING_OPSET_VERSION: Warning multiple opset versions specified, using highest.
Traceback (most recent call last):
  File "/opt/qairt/2.44.0.260225/lib/python/qti/aisw/converters/onnx/onnx_to_ir.py", line 702, in convert
    supported_version = self.translations.apply_method_to_op(src_type,
  File "/opt/qairt/2.44.0.260225/lib/python/qti/aisw/converters/common/converter_ir/translation.py", line 64, in apply_method_to_op
    translation = self.__get_translation(op_type)
  File "/opt/qairt/2.44.0.260225/lib/python/qti/aisw/converters/common/converter_ir/translation.py", line 38, in __get_translation
    raise KeyError("No translation registered for op type {}.".format(op_type))
KeyError: 'No translation registered for op type onnx_mysca

CalledProcessError: Command '['qnn-onnx-converter', '--input_network', 'work/mini_custom.onnx', '--output_path', 'work/mini_custom.cpp']' returned non-zero exit status 255.

现在添加自定义的算子

In [17]:
subprocess.run([
    "qnn-onnx-converter",
    "--input_network", "work/mini_custom.onnx",
    "--output_path", "work/mini_custom.cpp",
    "--op_package_config", "/workspace/work/op_package/MyScaleOpPackage/config/custom_Op.xml"
], check=True)

reEvalMacsParams: Re-evaluated graph statistics after IrOptimizer optimization passes:
reEvalMacsParams: Total MACs: 1
reEvalMacsParams: Total Params Count: 0


2026-04-05 11:41:41,950 - 278 - INFO - Input shape info 
2026-04-05 11:41:41,951 - 283 - WARNING - Can't simplify the model when custom ops, quantization overrides or dynamic input shapes are specified, converting without simplification.
2026-04-05 11:41:42,015 - 283 - WARNING - WARNING_OPSET_VERSION: Warning multiple opset versions specified, using highest.
2026-04-05 11:41:42,015 - 283 - WARNING - Skipping Native checker in Defer Loading and ONNX Simplification not invoked
2026-04-05 11:41:42,025 - 278 - INFO - Skipping quantization, no input_list provided
2026-04-05 11:41:42,025 - 278 - INFO - Saving QNN Model...
2026-04-05 11:41:42,025 - 278 - INFO - Model CPP saved at: work/mini_custom.cpp 
2026-04-05 11:41:42,025 - 283 - WARNING - No raw files found for Model. Saving Model BIN skipped.
2026-04-05 11:41:42,026 - 278 - INFO - Conversion complete!


CompletedProcess(args=['qnn-onnx-converter', '--input_network', 'work/mini_custom.onnx', '--output_path', 'work/mini_custom.cpp', '--op_package_config', '/workspace/work/op_package/MyScaleOpPackage/config/custom_Op.xml'], returncode=0)

把 QNN model lib 编出来

In [18]:
subprocess.run([
    "qnn-model-lib-generator",
    "-c", "work/mini_custom.cpp",
    "-t", "x86_64-linux-clang",
    "-o", "work/model_lib"
], check=True)

2026-04-05 11:45:45,861 -    INFO - qnn-model-lib-generator: Model cpp file path  : work/mini_custom.cpp
2026-04-05 11:45:45,861 -    INFO - qnn-model-lib-generator: Model bin file path  : None
2026-04-05 11:45:45,861 -    INFO - qnn-model-lib-generator: Library target       : [['x86_64-linux-clang']]
2026-04-05 11:45:45,861 -    INFO - qnn-model-lib-generator: Library name         : qnn_model
2026-04-05 11:45:45,861 -    INFO - qnn-model-lib-generator: Output directory     : work/model_lib
2026-04-05 11:45:45,861 - WARNING - qnn-model-lib-generator: Runtime will fail if .cpp needs data in .raw files when building model library
2026-04-05 11:45:45,861 -    INFO - qnn-model-lib-generator: Output library name  : qnn_model
2026-04-05 11:45:47,462 -    INFO - qnn-model-lib-generator: command : ['cd /workspace/tmp_5749 && export QNN_MODEL_LIB_NAME=libqnn_model.so && make CXX="clang++" -f Makefile.linux-x86_64']
2026-04-05 11:45:47,462 -    INFO - qnn-model-lib-generator: rc : 0
2026-04-05 1

CompletedProcess(args=['qnn-model-lib-generator', '-c', 'work/mini_custom.cpp', '-t', 'x86_64-linux-clang', '-o', 'work/model_lib'], returncode=0)

准备输入数据

In [19]:
import numpy as np
x = np.array([[-1.0, 0.5, 2.0]], dtype=np.float32)
x.tofile("input.raw")

先尝试不添加自定义op

In [31]:
subprocess.run([
    "qnn-net-run",
    "--model", "work/model_lib/x86_64-linux-clang/libqnn_model.so",
    "--input_list", "input.txt",
    "--output_dir", "./output",
    "--backend", "libQnnCpu.so",
    "--profiling_level", "detailed",
    "--debug"
], check=True)

qnn-net-run pid:32058
qnn-net-run build version: v2.44.0.260225143659
qnn-net-run log level is : QNN_LOG_LEVEL_ERROR
Processing inference input(s):
./input.raw
Composing Graphs
     1.6ms [ ERROR ] [QNN_CPU] Invalid OpPackage MyScale
[ ERROR ] QnnModel::addNode() validating node MyScale_0 failed.
[ ERROR ] model.addNode(QNN_OPCONFIG_VERSION_1, "MyScale_0", "MyScaleOpPackage", "MyScale", params_MyScale_0, 1, inputs_MyScale_0, 1, outputs_MyScale_0, 1 ) expected MODEL_NO_ERROR, got MODEL_GRAPH_OP_VALIDATION_ERROR
[ ERROR ] addNode_MyScale_0(mini_custom) expected MODEL_NO_ERROR, got MODEL_GRAPH_OP_VALIDATION_ERROR
     3.2ms [ ERROR ] Failed in composeGraphs()
     3.2ms [ ERROR ] ComposeGraphs Failed with error = 1


Graph Prepare failure



CalledProcessError: Command '['qnn-net-run', '--model', 'work/model_lib/x86_64-linux-clang/libqnn_model.so', '--input_list', 'input.txt', '--output_dir', './output', '--backend', 'libQnnCpu.so', '--profiling_level', 'detailed', '--debug']' returned non-zero exit status 14.

In [33]:
subprocess.run([
    "qnn-net-run",
    "--model", "work/model_lib/x86_64-linux-clang/libqnn_model.so",
    "--input_list", "input.txt",
    "--output_dir", "./output",
    "--backend", "libQnnCpu.so",
    "--op_packages", "work/op_package/MyScaleOpPackage/libs/x86_64-linux-clang/libMyScaleOpPackage.so:MyScaleOpPackageInterfaceProvider",
    "--profiling_level", "detailed",
    "--debug"
], check=True)

qnn-net-run pid:32672
qnn-net-run build version: v2.44.0.260225143659
qnn-net-run log level is : QNN_LOG_LEVEL_ERROR
Processing inference input(s):
./input.raw
Composing Graphs
Finalizing Graphs
Executing Graphs
Finished Executing Graphs


CompletedProcess(args=['qnn-net-run', '--model', 'work/model_lib/x86_64-linux-clang/libqnn_model.so', '--input_list', 'input.txt', '--output_dir', './output', '--backend', 'libQnnCpu.so', '--op_packages', 'work/op_package/MyScaleOpPackage/libs/x86_64-linux-clang/libMyScaleOpPackage.so:MyScaleOpPackageInterfaceProvider', '--profiling_level', 'detailed', '--debug'], returncode=0)

In [ ]:
y = np.fromfile("output/Result_0/output.raw", dtype=np.float32)
relu = np.fromfile("output/Result_0/_Relu_output_0.raw", dtype=np.float32)
print( x,relu, y)

[[-1.   0.5  2. ]] [0.  0.5 2. ] [0. 1. 4.]


In [ ]:
subprocess.run([
    "qnn-prof",
    "-m", "work/model_lib/libmini_custom.so",
    "-i", "input.raw",
    "-o", "output.raw",
    "--input_shapes", "1,3",
    "--input_dtypes", "float32",
    "--output_shapes", "1,3",
    "--output_dtypes", "float32"
], check=True)

In [40]:
subprocess.run([
    "qnn-profile-viewer",
    "--input_log", "output/qnn-profiling-data_0.log",
    "--output", "visual_profiling.csv",
    ], check=True)

Input Log File Location: output/qnn-profiling-data_0.log
Log File Created: Sun Apr  5 12:04:23 2026
Time Scale: 1e-06
Epoch Timestamp: 1775361863109926 Steady Clock Timestamp: 400346743085
Generated using: 
qnn-profile-viewer: v2.44.0.260225143659
qnn-net-run: v2.44.0.260225143659
Backend: v2.44.0.260225143659

Qnn Init/Prepare/Finalize/De-Init/Execute/Lib-Load Statistics:
------------------------------------------------------------
Init Stats:
-----------
    NetRun: 592 us

Compose Graphs Stats:
--------------
    NetRun: 564 us

Finalize Stats:
---------------
Graph 0 (mini_custom):
    NetRun: 25 us
    Backend (GRAPH_FINALIZE): 23 us
        __Relu: 7 us
        _MyScale_0: 3 us

De-Init Stats:
--------------
    NetRun: 501 us
    Backend (null): 0 us

Execute Stats (Overall):
------------------------
    NetRun IPS (includes IO and misc. time): 301.8412 inf/sec 

Execute Stats (Average):
------------------------
Total Inference Time: 
---------------------
Graph 0 (mini_custom):

CompletedProcess(args=['qnn-profile-viewer', '--input_log', 'output/qnn-profiling-data_0.log', '--output', 'visual_profiling.csv'], returncode=0)